# test-1: can an SLM predict a tiny MLP's final performance?
1. Train a tiny 2-H-1 MLP under many hyperparameter configs and log everything (weights before/after each step, gradients, activations).
2. Predict `final val loss` from **T1** hyperparameters only, and **T2** hyperparameters + early weights (step k).
3. Compare a gradient-boosting baseline with a fine-tuned small language model (SLM). Test configs are never seen in training.

On Kaggle: upload this folder (or add it as a dataset) and set `CODE_DIR` below.

In [ ]:
import os, sys
CODE_DIR = "."   # on Kaggle e.g. "/kaggle/input/test-1" ; outputs go to ./out
sys.path.insert(0, CODE_DIR)
os.makedirs("out", exist_ok=True)

N_CONFIGS, SEEDS, K = 200, (0, 1, 2), 10   # K = step at which T2 observes the weights
USE_SLM = True                              # needs `transformers` + a GPU (Kaggle: enable GPU, internet)

## 1. Generate the dataset

In [ ]:
from generate import generate
runs, steps, fwd = generate(N_CONFIGS, SEEDS, out_dir="out")
print(len(runs), "runs |", len(steps), "step rows |", len(fwd), "forward rows")
runs.head()

Look at one run: weights **before** the step, the gradient, and weights **after** the step.

In [ ]:
rid = runs.run_id[0]
cols = ["step", "before_W1_00", "grad_W1_00", "after_W1_00", "batch_loss", "val_loss", "val_acc"]
steps[steps.run_id == rid][cols].head()

Probe points through the network: inputs `x`, mixes `z`, activations `a`, output `y_hat`.

In [ ]:
fwd[(fwd.run_id == rid) & (fwd.step == 0)].head()

Noise floor: how much does the same config vary across seeds? No predictor can beat this.

In [ ]:
import numpy as np
runs["log_loss"] = np.log10(runs.final_val_loss)
print("mean std across seeds (log10 loss):", runs.groupby("config_id").log_loss.std().mean())
print("std across configs   (log10 loss):", runs.log_loss.std())

## 2. Baseline (gradient boosting) on T1 and T2

In [ ]:
import pandas as pd
from tasks import build_task, split, score, gbm_baseline

results = {}
for task in ["T1", "T2"]:
    X, texts, y, groups = build_task(runs, steps, task, K)
    tr, te = split(groups)
    results[f"GBM {task}"] = score(y[te], gbm_baseline(X, y, tr, te))
pd.DataFrame(results).T

## 3. SLM on T1 and T2

In [ ]:
if USE_SLM:
    from slm import fit_predict
    for task in ["T1", "T2"]:
        X, texts, y, groups = build_task(runs, steps, task, K)
        tr, te = split(groups)
        pred = fit_predict([texts[i] for i in tr], y[tr], [texts[i] for i in te])
        results[f"SLM {task}"] = score(y[te], pred)
    print(texts[0])   # example input text
pd.DataFrame(results).T

Reading the table: lower `MAE` and higher `Spearman` are better. If **T2 beats T1**, early internal weights carry information beyond the hyperparameters.